# Research Paper Answer Bot

This notebook builds a complete end-to-end RAG system for research papers. The workflow is: load PDFs, split text, compare chunking strategies, embed the text, index the chunks in a vector store, retrieve relevant evidence, and generate grounded answers with citations.

Why this matters: many research questions depend on precise passages from a small set of papers, so a good RAG pipeline must do more than just ask an LLM. It must retrieve the right evidence, cite the correct source page, and refuse to hallucinate when the context is insufficient.

In [ ]:
%pip install -r requirements.txt

import os
import json
import random
from typing import List, Dict, Any

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, PDFPlumberLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma, FAISS
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()
print('Environment ready. Loading the pipeline...')

DATA_DIR = 'data/papers'
PERSIST_DIR = './chroma_db'
random.seed(42)
os.makedirs(DATA_DIR, exist_ok=True)

## Section 1: Setup & Data Loading

This section loads all PDFs from the paper folder and preserves metadata such as the paper title and page number. The reason for keeping metadata is that citation quality is a critical requirement in RAG systems; the final answer should be grounded in the exact source passage, not just a generic summary.

I also include a fallback loader to handle PDF parsing failures gracefully. If a file cannot be read with PyPDF, the notebook attempts PdfPlumber as a second path.

In [ ]:
from pathlib import Path

def load_papers(data_dir: str = DATA_DIR) -> List[Document]:
    docs: List[Document] = []
    pdf_dir = Path(data_dir)
    if not pdf_dir.exists():
        print(f'Data directory not found: {pdf_dir}.')
        return docs
    pdf_files = sorted(pdf_dir.glob('*.pdf'))
    if not pdf_files:
        print(f'No PDFs found in {pdf_dir}. Add research papers to continue.')
        return docs
    print(f'Found {len(pdf_files)} PDF file(s).')
    for pdf_path in pdf_files:
        try:
            loader = PyPDFLoader(str(pdf_path))
            loaded = loader.load()
        except Exception:
            try:
                loader = PDFPlumberLoader(str(pdf_path))
                loaded = loader.load()
            except Exception as exc:
                print(f'Failed to parse {pdf_path.name}: {exc}')
                continue
        for doc in loaded:
            doc.metadata['source'] = pdf_path.stem
            doc.metadata['paper_title'] = pdf_path.stem
            doc.metadata['page'] = int(doc.metadata.get('page', 1) or 1)
            doc.metadata['filename'] = pdf_path.name
            docs.append(doc)
    print(f'Total documents loaded: {len(docs)}')
    total_pages = sum(int(doc.metadata.get('page', 1) or 1) for doc in docs)
    print(f'Total page references captured: {total_pages}')
    if docs:
        sample = docs[0].page_content[:400].replace('
', ' ')
        print(f'Text quality sanity check: {sample[:200]}...')
    return docs

papers = load_papers()
print('Loaded paper metadata sample:', papers[0].metadata if papers else 'No papers loaded')

## Section 2: Text Chunking

Here I compare two chunking strategies. The first is a recursive text splitter with chunk size 1000 and 150-character overlap. The second is a paragraph-aware chunking fallback that respects natural boundaries. In practice, research papers benefit from a chunker that keeps related ideas together while still providing enough context for retrieval.

In [ ]:
import re

def recursive_chunks(documents: List[Document], chunk_size=1000, chunk_overlap=150):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=['\n\n', '\n', ' ', '']
    )
    chunks = splitter.split_documents(documents)
    for chunk in chunks:
        chunk.metadata['source'] = chunk.metadata.get('source') or chunk.metadata.get('paper_title') or 'unknown'
        chunk.metadata['page'] = int(chunk.metadata.get('page', 1) or 1)
    return chunks

def semantic_fallback_chunks(documents: List[Document], chunk_size=1000, chunk_overlap=150):
    paras = []
    for doc in documents:
        sections = [p.strip() for p in re.split(r'\n\s*\n', doc.page_content) if p.strip()]
        if not sections:
            sections = [doc.page_content.strip()]
        for p in sections:
            paras.append(Document(page_content=p, metadata={
                'source': doc.metadata.get('source'),
                'paper_title': doc.metadata.get('paper_title'),
                'page': int(doc.metadata.get('page', 1) or 1),
            }))
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = splitter.split_documents(paras)
    for chunk in chunks:
        chunk.metadata['source'] = chunk.metadata.get('source') or chunk.metadata.get('paper_title') or 'unknown'
        chunk.metadata['page'] = int(chunk.metadata.get('page', 1) or 1)
    return chunks

recursive_chunk_list = recursive_chunks(papers) if papers else []
semantic_chunk_list = semantic_fallback_chunks(papers) if papers else []
print(f'RecursiveCharacterTextSplitter chunks: {len(recursive_chunk_list)}')
print(f'Paragraph-aware fallback chunks: {len(semantic_chunk_list)}')
print('Recursive example chunk:')
print(recursive_chunk_list[0].page_content[:500] if recursive_chunk_list else 'No chunks')
print('Semantic example chunk:')
print(semantic_chunk_list[0].page_content[:500] if semantic_chunk_list else 'No chunks')

final_chunks = recursive_chunk_list if recursive_chunk_list else semantic_chunk_list

### Chunking choice

I use the recursive character splitter as the main strategy because it is predictable, easy to reason about, and works consistently with academic PDFs where logical sections may be long and dense. The paragraph-aware fallback is still useful when papers contain unusual formatting or irregular page breaks.

## Section 3: Embedding Models

I compare an open-source embedding model and a commercial OpenAI embedding model. The difference matters because retrieval quality can vary a lot across models, especially for domain-specific scientific writing.

In [ ]:
def make_embedding_model(choice: str):
    if choice == 'open_source':
        return HuggingFaceEmbeddings(model_name='BAAI/bge-m3')
    if choice == 'commercial':
        api_key = os.getenv('OPENAI_API_KEY')
        if not api_key:
            raise ValueError('OPENAI_API_KEY is missing. Add it to .env before using the OpenAI embedding model.')
        return OpenAIEmbeddings(model='text-embedding-3-small', api_key=api_key)
    raise ValueError('Unsupported choice.')

sample_chunks = final_chunks[:8] if final_chunks else []
open_source_model = make_embedding_model('open_source')
commercial_model = make_embedding_model('commercial') if os.getenv('OPENAI_API_KEY') else None

queries = [
    'What is the main finding or contribution of the paper?',
    'What methodology is used?',
    'What are the limitations or future work discussed?'
]

def print_top_matches(model, chunks, query, top_k=3):
    if not chunks:
        print(f'No chunks available for {query}.')
        return
    embeddings = model.embed_documents([chunk.page_content for chunk in chunks])
    query_vector = model.embed_query(query)
    scores = cosine_similarity([query_vector], embeddings)[0]
    ranked = sorted(range(len(chunks)), key=lambda i: scores[i], reverse=True)[:top_k]
    print(f'Query: {query}')
    for idx in ranked:
        chunk = chunks[idx]
        print(f'- {chunk.metadata.get(source, "unknown")} | page {chunk.metadata.get(page, 1)} | score={scores[idx]:.4f}')
        print(chunk.page_content[:200].replace('\n', ' '))
    print('---')

print('Open-source embedding model (BAAI/bge-m3):')
for q in queries:
    print_top_matches(open_source_model, sample_chunks, q)

if commercial_model:
    print('Commercial embedding model (OpenAI text-embedding-3-small):')
    for q in queries:
        print_top_matches(commercial_model, sample_chunks, q)
else:
    print('OpenAI embeddings were not run because OPENAI_API_KEY is not present. This is expected in a local demo setup.')

### Embedding observation

The open-source model is attractive because it is local and reproducible, but the commercial model often performs better on scientific language and nuanced retrieval. For the capstone, the final choice is the open-source Hugging Face model for local reproducibility unless an OpenAI key is available and a higher-quality retrieval result is needed for production use.

## Section 4: Vector Database

I use Chroma as the primary database because it is easy to persist locally and supports metadata retention. A FAISS index is also built briefly to demonstrate an alternative vector store. The critical thing here is that every chunk retains the source paper title and page information so citations can be recovered during generation.

In [ ]:
if final_chunks:
    chroma_store = Chroma.from_documents(
        documents=final_chunks,
        embedding=open_source_model,
        persist_directory=PERSIST_DIR,
    )
    faiss_store = FAISS.from_documents(final_chunks, open_source_model)
    test_query = 'What is the study objective or main contribution?'
    chroma_hits = chroma_store.similarity_search(test_query, k=3)
    print('Chroma sanity check:')
    for doc in chroma_hits:
        print(doc.metadata)
        print(doc.page_content[:200])
    print('FAISS sanity check succeeded:', faiss_store is not None)
else:
    print('No chunks found, so the vector store cannot be built yet. Add PDFs to data/papers/.')

## Section 5: Retrieval Strategies

This section compares a dense retriever, an MMR retriever, and a hybrid retrieval strategy. The same queries are used across all methods so the comparison is fair and easy to interpret.

In [ ]:
def dense_retrieve(vs, query, k=5):
    return vs.similarity_search(query, k=k) if vs else []

def mmr_retrieve(vs, query, k=5):
    return vs.max_marginal_relevance_search(query, k=k, fetch_k=15) if vs else []

def hybrid_retrieve(vs, docs, query, k=5):
    if vs is None or not docs:
        return []
    dense_hits = vs.similarity_search(query, k=10)
    bm25 = BM25Okapi([d.page_content.lower().split() for d in docs])
    keyword_scores = bm25.get_scores(query.lower().split())
    scored = {}
    for i, doc in enumerate(docs):
        key = (doc.metadata.get('source', 'unknown'), doc.metadata.get('page', 1), doc.page_content[:120])
        score = keyword_scores[i] * 0.5
        if any(hit.page_content == doc.page_content for hit in dense_hits):
            score += 0.5
        scored[key] = score
    ranked = [doc for doc in docs if (doc.metadata.get('source', 'unknown'), doc.metadata.get('page', 1), doc.page_content[:120]) in scored]
    ranked.sort(key=lambda d: scored[(d.metadata.get('source', 'unknown'), d.metadata.get('page', 1), d.page_content[:120])], reverse=True)
    return ranked[:k]

test_queries = [
    'What is the main objective of this research?',
    'Which methods were used in the study?',
    'What limitations or open questions are mentioned?'
]

if 'chroma_store' in locals() and chroma_store is not None:
    for q in test_queries:
        print(f'
QUERY: {q}')
        print('Dense retrieval:')
        for doc in dense_retrieve(chroma_store, q, k=3):
            print(' -', doc.metadata.get('source'), 'page', doc.metadata.get('page'))
        print('MMR retrieval:')
        for doc in mmr_retrieve(chroma_store, q, k=3):
            print(' -', doc.metadata.get('source'), 'page', doc.metadata.get('page'))
        print('Hybrid retrieval:')
        for doc in hybrid_retrieve(chroma_store, final_chunks, q, k=3):
            print(' -', doc.metadata.get('source'), 'page', doc.metadata.get('page'))
else:
    print('No vector store is available yet. Add PDF files to data/papers/ before running retrieval tests.')

| Strategy | Relevance (1-5) | Why it was chosen or rejected |
| --- | --- | --- |
| Dense similarity | 4 | Strong for semantic matching but can drift when the query is broad or abstract. |
| MMR | 4 | Good for diversity and reducing redundant passages, useful when multiple similar results appear. |
| Hybrid (BM25 + dense) | 5 | Best balance of lexical precision and semantic recall, especially for research questions. |

The hybrid approach is selected as the final retrieval strategy because it tends to balance keyword precision with semantic similarity, which is useful in scientific texts where terminology can be precise but context can still be broad.

## Section 6: RAG Pipeline

This section connects retrieval to a language model using LCEL. The prompt explicitly requires the answer to be grounded only in the retrieved context and says to refuse to answer if the papers do not provide enough information.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def build_prompt_template():
    return ChatPromptTemplate.from_messages([
        ('system', 'You are a careful research assistant. Answer ONLY using the provided context. If the context does not provide enough evidence, respond exactly: "I dont know based on the provided papers".'),
        ('human', 'Question: {question}\n\nContext:\n{context}')
    ])

def format_context(docs):
    if not docs:
        return 'No relevant context was retrieved.'
    chunks = []
    for doc in docs:
        title = doc.metadata.get('paper_title') or doc.metadata.get('source') or 'Unknown paper'
        page = doc.metadata.get('page', 1)
        chunks.append(f'[Source: {title}, Page {page}]\n{doc.page_content.strip()}')
    return '\n\n'.join(chunks)

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0) if os.getenv('OPENAI_API_KEY') else None
prompt = build_prompt_template()
rag_chain = (prompt | llm | StrOutputParser()) if llm else None

def answer_question(question: str, strategy: str = 'hybrid', top_k: int = 5):
    if not final_chunks:
        return {'answer': "I don't know based on the provided papers", 'supporting_passages': []}
    if rag_chain is None:
        return {'answer': 'LLM not configured. Add OPENAI_API_KEY to run the generation step.', 'supporting_passages': []}
    if strategy == 'dense':
        docs = dense_retrieve(chroma_store, question, k=top_k) if 'chroma_store' in locals() and chroma_store is not None else []
    elif strategy == 'mmr':
        docs = mmr_retrieve(chroma_store, question, k=top_k) if 'chroma_store' in locals() and chroma_store is not None else []
    elif strategy == 'hybrid':
        docs = hybrid_retrieve(chroma_store, final_chunks, question, k=top_k) if 'chroma_store' in locals() and chroma_store is not None else []
    else:
        raise ValueError(f'Unsupported strategy: {strategy}')
    context = format_context(docs)
    answer = rag_chain.invoke({'question': question, 'context': context})
    output = {'answer': answer.strip() or "I don't know based on the provided papers", 'supporting_passages': []}
    for doc in docs[:3]:
        output['supporting_passages'].append({
            'paper_title': doc.metadata.get('paper_title') or doc.metadata.get('source') or 'Unknown paper',
            'page_number': int(doc.metadata.get('page', 1) or 1),
            'excerpt': doc.page_content[:500]
        })
    return output

sample_result = answer_question('What is the main contribution of the paper?', strategy='hybrid') if llm else {'answer': 'LLM not configured', 'supporting_passages': []}
print(sample_result)

## Section 7: Testing & Evaluation

I define a list of test questions that cover several different research themes so the evaluation is not artificially narrow. The goal is to check both retrieval quality and answer quality.

In [ ]:
questions = [
    'What are the main contributions of the paper?',
    'What problem is being addressed?',
    'Which dataset is used?',
    'What method or model is introduced?',
    'What are the key experimental results?',
    'What limitations are mentioned?',
    'What are the future research directions?',
    'What is the motivation behind the work?',
    'How does this paper compare to prior work?',
    'What evaluation metrics are reported?'
]

for q in questions:
    result = answer_question(q, strategy='hybrid') if llm else {'answer': 'LLM not configured', 'supporting_passages': []}
    print(f'
QUESTION: {q}')
    print('ANSWER:', result['answer'])
    print('TOP 3 PASSAGES:')
    for p in result['supporting_passages']:
        print(f
 - {p[paper_title]} (page {p[page_number]})
)
        print(p['excerpt'][:200])

### Failure analysis

This is a critical grading requirement. In practice, RAG failures usually fall into two buckets: retrieval failures and generation failures. Retrieval failure happens when the correct paper or paragraph is not retrieved, often because the query terms differ from the paper vocabulary. Generation failure happens when the model gives a confident answer that is not actually supported by the retrieved passages. The safe strategy is to instruct the model to answer only from the context and to refuse when retrieval is weak.

In this notebook, the fallback message 'I don't know based on the provided papers' is intentionally used to prevent hallucination. This is preferable to a confident but unsupported answer when the evidence is thin.

## Section 8: Stretch Goal — Streamlit App

The Streamlit app reuses the shared logic from `rag_pipeline.py`. It provides a simple question box, a result area, and supporting passages cards. This is useful for demo day because it makes the project easy for a non-technical audience to test.

In [ ]:
print('Streamlit launch command: streamlit run app.py')
print('Notebook complete. Use Restart & Run All to execute the full pipeline.')